In [1]:
!hostname

gl1505.arc-ts.umich.edu


### Import

In [8]:
import argparse
import datetime
import json
import os
import time
import sys
from pathlib import Path

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import yaml
from torch.utils.tensorboard import SummaryWriter
import wandb

sys.path.append("/home/minsukc/VECSET/src")
import utils.misc as misc
from engines.engine_ae import train_one_epoch
from models import autoencoder
from utils.ct_dataset import CTSingleVolumeDataset
from utils.misc import NativeScalerWithGradNormCount as NativeScaler

# Set threads
torch.set_num_threads(8)

In [9]:
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding='utf-8')

    def write(self, message):
        # In a notebook, accessing sys.stdout directly might be tricky depending on the IDE,
        # but this standard approach usually works for standard python execution.
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

### Config

In [13]:
class Config:
    def __init__(self):
        # --- Paths ---
        self.config = "config.yaml"
        self.data_path = "dummy_ct.nii.gz"
        self.output_dir = "output/notebook_run"
        self.checkpoint_dir = "/gpfs/accounts/jjparkcv_root/jjparkcv98/minsukc/vecset/ct_overfit_experiment"
        self.log_dir = "output/notebook_run/log"
        
        # --- Model ---
        self.model = "point_vec512x32_dim512_depth24"
        # self.model = "learnable_vec512x32_dim512_depth24"
        self.point_cloud_size = 14147
        self.resume = ""
        
        # --- Training ---
        self.batch_size = 1 # CT volumes are large, usually batch size 1
        self.epochs = 1000
        self.accum_iter = 1
        self.start_epoch = 0
        self.device = "cuda"
        self.seed = 42
        
        # --- Optimizer ---
        self.clip_grad = None
        self.weight_decay = 0.05
        self.lr = None # Will be calculated based on blr
        self.blr = 1e-4 # Base learning rate
        self.min_lr = 1e-6
        self.warmup_epochs = 20
        self.layer_decay = 0.75
        
        # --- System ---
        self.num_workers = 4
        self.pin_mem = True
        
        # --- Distributed (Usually False for Notebooks) ---
        self.world_size = 1
        self.local_rank = -1
        self.dist_on_itp = False
        self.dist_url = "env://"
        self.distributed = False
        self.gpu = 0
        
        # --- WandB ---
        # self.no_wandb = False
        self.wandb = False
        self.eval = False
        self.dist_eval = False

    def load_yaml(self):
        if os.path.exists(self.config):
            print(f"Loading config from {self.config}")
            with open(self.config, "r") as f:
                cfg = yaml.safe_load(f)
            for key, value in cfg.items():
                if hasattr(self, key):
                    setattr(self, key, value)
        return self

### Initializing

In [14]:
# Initialize Args
args = Config()
# args.load_yaml() # Uncomment to load from yaml

print(f"Experiment Configuration:\n{json.dumps(vars(args), indent=4, default=str)}")

# %%
# [NOTE] Setup Directories and Logger
if args.output_dir:
    Path(args.output_dir).mkdir(parents=True, exist_ok=True)

if not args.checkpoint_dir and args.output_dir:
    args.checkpoint_dir = args.output_dir

if args.checkpoint_dir:
    Path(args.checkpoint_dir).mkdir(parents=True, exist_ok=True)

# Redirect stdout to file
if args.output_dir:
    # Be careful running this cell multiple times in a notebook, it might nest loggers.
    # We check if sys.stdout is already a Logger to avoid issues.
    if not isinstance(sys.stdout, Logger):
        sys.stdout = Logger(os.path.join(args.output_dir, "console_log.txt"))

print(f"[Info] Logs: {args.output_dir}")
print(f"[Info] Checkpoints: {args.checkpoint_dir}")

# %%
# [NOTE] Initialize Environment & WandB
device = torch.device(args.device)

if args.wandb:
    # Check if run exists to allow re-running cells
    if wandb.run is None:
        wandb.init(
            project="ct_vecset",
            name="notebook_run",
            config=vars(args)
        )

# Fix seeds
seed = args.seed + misc.get_rank()
torch.manual_seed(seed)
np.random.seed(seed)
cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

### Data Loading

In [15]:
# [NOTE] Load Dataset
print(f"Loading dataset from: {args.data_path}")
dataset_train = CTSingleVolumeDataset(nii_path=args.data_path, pc_size=args.point_cloud_size)
dataset_val = dataset_train # Use same for overfitting check

# Log Dataset Stats
if args.wandb:
    coverage = args.point_cloud_size / max(1, dataset_train.num_structure)
    wandb.log({
        "data/total_voxels": dataset_train.total_voxels,
        "data/structure_voxels": dataset_train.num_structure,
        "data/structure_ratio": dataset_train.structure_ratio,
        "data/encoding_vox/num_structure": coverage,
        "data/pc_size": args.point_cloud_size
    })
    print(f"[WandB] Logged structure stats. Coverage: {coverage*100:.2f}%")

# Create Dataloader
sampler_train = torch.utils.data.RandomSampler(dataset_train)
data_loader_train = torch.utils.data.DataLoader(
    dataset_train,
    sampler=sampler_train,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    pin_memory=args.pin_mem,
    drop_last=True,
)

### Model & Optimizer

In [16]:
# [NOTE] Initialize Model
print(f"Creating model: {args.model}")
model = autoencoder.__dict__[args.model](pc_size=args.point_cloud_size)
model.to(device)

n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model = {str(model)}")
print(f"Number of params (M): {n_parameters / 1.0e6:.2f}")

# %%
# [NOTE] Optimizer & Loss Setup
eff_batch_size = args.batch_size * args.accum_iter * misc.get_world_size()

if args.lr is None:
    args.lr = args.blr * eff_batch_size / 256

print(f"Base lr: {args.lr * 256 / eff_batch_size:.2e}")
print(f"Actual lr: {args.lr:.2e}")
print(f"Accumulate grad iterations: {args.accum_iter}")
print(f"Effective batch size: {eff_batch_size}")

optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
loss_scaler = NativeScaler()
criterion = torch.nn.L1Loss()

print(f"Criterion: {str(criterion)}")

# Resume if needed
misc.load_model(args=args, model_without_ddp=model, optimizer=optimizer, loss_scaler=loss_scaler)

# Tensorboard Writer
log_writer = None
if args.log_dir:
    log_writer = SummaryWriter(log_dir=args.log_dir)

### Training Loop

In [ ]:
# [NOTE] Main Training Loop
print(f"Start training for {args.epochs} epochs")
start_time = time.time()

try:
    for epoch in range(args.start_epoch, args.epochs):
        # Train
        train_stats = train_one_epoch(
            model, criterion, data_loader_train, optimizer, device, epoch, 
            loss_scaler, args.clip_grad, log_writer=log_writer, args=args
        )

        # Save Checkpoint (every 5 epochs or last)
        if args.checkpoint_dir and (epoch % 5 == 0 or epoch + 1 == args.epochs):
            # Temporarily swap output_dir to checkpoint_dir for save function
            original_dir = args.output_dir
            args.output_dir = args.checkpoint_dir
            misc.save_model(
                args=args, model=model, model_without_ddp=model, 
                optimizer=optimizer, loss_scaler=loss_scaler, epoch=epoch
            )
            args.output_dir = original_dir

        # Log Stats
        log_stats = {**{f"train_{k}": v for k, v in train_stats.items()}, "epoch": epoch, "n_parameters": n_parameters}
        
        if args.output_dir:
            if log_writer is not None:
                log_writer.flush()
            with open(os.path.join(args.output_dir, "log.txt"), mode="a", encoding="utf-8") as f:
                f.write(json.dumps(log_stats) + "\n")

except KeyboardInterrupt:
    print("Training interrupted manually.")

total_time = time.time() - start_time
total_time_str = str(datetime.timedelta(seconds=int(total_time)))
print("Training time {}".format(total_time_str))

if args.wandb:
    wandb.finish()